# New Slice-Aware Scheduler Sanity Test

This notebook checks that the real multi-gNB scheduler is queue-aware, slice-budget-aware, and demand-controlled.

Role separation in this test:
- Scheduler: PRB allocation only for currently attached UEs.
- A3/HRL: mobility and load-balancing control through offsets.

The key failure this catches is the old behavior where a controlled snapshot such as eMBB `0.88 / 0.18 / 0.48` collapsed to `1.00 / 1.00 / 1.00` after a real scheduler tick.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Avoid matplotlib cache warnings on machines where ~/.config is read-only.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from global_ppo_3gnb_env import GlobalPPO3GNBEnv

np.set_printoptions(precision=3, suppress=True)

In [ ]:
def load_matrix(env):
    return np.asarray(env._load_matrix(), dtype=float)


def kpi_frame(env):
    rows = []
    for (gnb_id, slice_type), kpi in sorted(env.base_env._last_info.get("slice_kpis", {}).items()):
        row = {"gnb_id": gnb_id, "slice_type": slice_type}
        row.update(kpi)
        rows.append(row)
    return pd.DataFrame(rows)


def print_loads(label, matrix):
    print(label)
    print(np.round(matrix, 3))

## 1. Pure Scheduler Tick, No A3 Offset Application

This advances only the base environment once after snapshot initialization. It isolates scheduler/load behavior from mobility decisions.

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=11,
    snapshot_scenario="embb_g0_offload",
    local_steps_per_global=1,
    global_steps_per_episode=2,
    radio_substeps=1,
)

obs, info = env.reset(seed=11)
initial = load_matrix(env)
print_loads("Initial snapshot load matrix", initial)

_obs, _reward, _terminated, _truncated, base_info = env.base_env.step(0)
after_scheduler = load_matrix(env)
print_loads("After one base-env scheduler step", after_scheduler)

assert not np.allclose(after_scheduler, np.ones_like(after_scheduler)), "Scheduler collapsed all loads to 1.00"
assert after_scheduler[0, 0] >= 0.75, "High eMBB gNB0 load should remain high"
assert after_scheduler[1, 0] <= 0.35, "Low eMBB gNB1 load should remain low/moderate"
assert after_scheduler[2, 0] <= 0.75, "Medium eMBB gNB2 load should not saturate"

print("Handovers during pure scheduler test:", len(env.base_env.handover_events))
display(kpi_frame(env))

env.close()

## 2. Slice Budget Checks

Every slice should report used PRBs at or below its own budget. This verifies that eMBB, URLLC, and mMTC are not borrowing the full gNB pool from each other.

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=12,
    snapshot_scenario="multi_slice_multi_gnb_congestion",
    local_steps_per_global=1,
    global_steps_per_episode=2,
    radio_substeps=1,
)

obs, info = env.reset(seed=12)
env.base_env.step(0)
kpis = kpi_frame(env)
display(kpis)

assert (kpis["used_prbs"] <= kpis["budget_prbs"]).all(), "A slice used more PRBs than its budget"
assert (kpis["load"] <= 1.0).all(), "Normalized slice load exceeded 1.0"
assert set(kpis["slice_type"]) == {"eMBB", "URLLC", "mMTC"}, "Expected all three slice types"

print_loads("Load matrix after budget-bounded scheduling", load_matrix(env))
env.close()

## 3. Neutral Upper Step With A3 Enabled

This allows the upper environment to compute neutral lower-agent offsets before the scheduler tick. Any UE placement changes here are from A3/HRL logic, not the scheduler.

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=11,
    snapshot_scenario="embb_g0_offload",
    local_steps_per_global=1,
    global_steps_per_episode=2,
    radio_substeps=1,
)

obs, info = env.reset(seed=11)
print_loads("Initial snapshot load matrix", load_matrix(env))

neutral_action = np.zeros(env.action_space.shape, dtype=np.float32)
obs, reward, terminated, truncated, info = env.step(neutral_action)
print_loads("After one neutral upper step", load_matrix(env))
print("Handovers during neutral upper step:", info["handover_count"])
display(kpi_frame(env))

assert not np.allclose(load_matrix(env), np.ones_like(load_matrix(env))), "Neutral upper step collapsed all loads to 1.00"
env.close()

## 4. Empty Queue Stop Rule

This directly checks the proportional-fair scheduler stop condition: with no queued demand, it must not allocate fake PRBs via `np.argmax`.

In [ ]:
from schedulers import ProportionalFair
from channel_models import MCSCodeset

env = GlobalPPO3GNBEnv(seed=13, snapshot_scenario="all_neutral", radio_substeps=1)
env.reset(seed=13)
ues = env.base_env.get_all_ues()[:3]

for ue in ues:
    ue.queue = 0
    ue.th = 0.0
    ue.estimate_snr(np.full(50, 15.0))
    ue.prbs = 0
    ue.bits = 0

scheduler = ProportionalFair(MCSCodeset(), granularity=2, slot_length=env.base_env.step_dt)
scheduler.allocate(ues, 50, slice_type="eMBB")

assert sum(int(ue.prbs) for ue in ues) == 0, "Empty queues received fake PRBs"
print("Empty queue stop rule OK: no PRBs allocated.")
env.close()